# 第 44 课｜AkShare A股数据契约与时点

本 Notebook 对应「365 天量化金融」第 44 课。课程方法来自本地完整视频、课件截图和本地 Whisper 口播整理；这里把它重建为默认离线、可核对、可重复执行的量化教学实现。

- 图文口播：`/Users/mojiawen/Documents/学习资料归档/_暂存/pam-video-captures/365-days-quant-finance/44_youtube-7slVGsPS9_s/index.html`
- 环境：`pkm-quant`
- 数据：离线构造的 A 股接口快照；非真实证券或财报数据
- 状态：可运行学习稿；不代表策略收益承诺

> 网络接口和研究计算分离：Notebook 默认不请求第三方服务。真实接口只作为适配层示意，计算使用带来源声明的本地快照，避免限流、字段变化和历史数据重算破坏复现。

## 本节量化问题

AkShare 同时覆盖行情、指数成分和财务数据，但三者的时间语义不同：

- 行情要记录 `adjust=qfq/hfq/none`；
- 指数权重要记录 `effective_date`；
- 财报要区分 `report_period`、`published_at` 和回测可用的 `available_at`。

本节用离线快照演示适配层如何把中文字段统一为稳定 schema。

In [1]:
from pathlib import Path
import json
import os
import platform
import sys
import tempfile
import timeit

# Matplotlib 首次运行需要写字体缓存；放到可写临时目录，避免污染用户目录。
MPL_CACHE_DIR = Path(tempfile.gettempdir()) / "pkm-quant-matplotlib"
MPL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPL_CACHE_DIR))

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import seaborn as sns

# 从本机字体中选择支持中文的字体，避免标题显示成方框。
preferred_chinese_fonts = [
    "PingFang SC",
    "Arial Unicode MS",
    "Hiragino Sans GB",
    "Heiti SC",
]
available_font_names = {font.name for font in font_manager.fontManager.ttflist}
chinese_font = next(
    (name for name in preferred_chinese_fonts if name in available_font_names),
    "DejaVu Sans",
)
plt.rcParams["font.sans-serif"] = [chinese_font, "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

def find_project_root():
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "实验" / "p2-python-toolchain",
        cwd / "p2-python-toolchain",
        *cwd.parents,
    ]
    for candidate in candidates:
        if (candidate / "课程辅助设计.md").exists():
            return candidate
    raise FileNotFoundError("请从 p2-python-toolchain 或量化交易项目目录启动 JupyterLab")

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data" / "fixtures"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_SEED = 365

print("Python:", sys.version.split()[0])
print("解释器:", sys.executable)
print("项目:", PROJECT_ROOT)
print("绘图字体:", chinese_font)

Python: 3.11.15
解释器: /opt/homebrew/Caskroom/miniforge/base/envs/pkm-quant/bin/python
项目: /Users/mojiawen/Documents/学习资料归档/_暂存/personal_knowledge_manager/量化交易/实验/p2-python-toolchain
绘图字体: PingFang SC


## 1. 适配器负责字段翻译，不让研究代码依赖网站列名

`rename` 把供应商中文字段转换为统一字段；`assign` 加入代码、复权和来源。后续计算只读取标准列，即使接口列名变化，也只修改适配器。

In [2]:
# 构造 AkShare 风格中文响应，再规范为统一日线契约。
raw_ak = (
    pd.read_csv(DATA_DIR / "daily_prices.csv", parse_dates=["date"])
    .query("symbol == 'BANK_A'")
    .head(8)
    .rename(columns={
        "date": "日期", "open": "开盘", "high": "最高", "low": "最低",
        "close": "收盘", "volume": "成交量",
    })
)

def normalize_ak_daily(frame, symbol, adjust):
    mapping = {"日期": "date", "开盘": "open", "最高": "high", "最低": "low",
               "收盘": "close", "成交量": "volume"}
    result = frame.rename(columns=mapping)[list(mapping.values())].copy()
    result["symbol"] = symbol
    result["adjustment"] = adjust
    result["source"] = "akshare-style-offline"
    return result.sort_values("date").reset_index(drop=True)

bars = normalize_ak_daily(raw_ak, "DEMO_A", "qfq")
display(raw_ak.head(3))
display(bars.head(3))
assert bars.columns.tolist() == [
    "date", "open", "high", "low", "close", "volume",
    "symbol", "adjustment", "source",
]
assert bars["adjustment"].eq("qfq").all()

,日期,symbol,sector,开盘,最高,最低,收盘,adj_close,成交量
0,2023-01-03,BANK_A,金融,11.7051,11.7522,11.6747,11.7218,11.7218,2822797
1,2023-01-04,BANK_A,金融,11.7142,11.7878,11.6199,11.6934,11.6934,1999548
2,2023-01-05,BANK_A,金融,11.6517,11.7478,11.5905,11.6864,11.6864,1974821


,date,open,high,low,close,volume,symbol,adjustment,source
0,2023-01-03,11.7051,11.7522,11.6747,11.7218,2822797,DEMO_A,qfq,akshare-style-offline
1,2023-01-04,11.7142,11.7878,11.6199,11.6934,1999548,DEMO_A,qfq,akshare-style-offline
2,2023-01-05,11.6517,11.7478,11.5905,11.6864,1974821,DEMO_A,qfq,akshare-style-offline


## 2. 指数成分要按生效日期做 point-in-time 选择

`effective_date <= as_of_date` 表示只使用当时已经生效的权重。下面同一股票在两个季度有不同权重；如果直接取最新行，就会把未来调仓信息带回过去。

In [3]:
# 选择研究日当时已生效的最新一组指数权重。
weights = pd.DataFrame({
    "effective_date": pd.to_datetime([
        "2024-01-01", "2024-01-01", "2024-04-01", "2024-04-01"
    ]),
    "symbol": ["DEMO_A", "DEMO_B", "DEMO_A", "DEMO_B"],
    "weight": [0.60, 0.40, 0.45, 0.55],
})
as_of_date = pd.Timestamp("2024-03-15")
valid_date = weights.loc[weights["effective_date"].le(as_of_date), "effective_date"].max()
point_in_time_weights = weights.loc[weights["effective_date"].eq(valid_date)].copy()

display(point_in_time_weights)
assert point_in_time_weights["weight"].sum() == 1.0
assert point_in_time_weights.set_index("symbol").loc["DEMO_A", "weight"] == 0.60

,effective_date,symbol,weight
0,2024-01-01,DEMO_A,0.6
1,2024-01-01,DEMO_B,0.4


## 3. 累计利润表不能直接当单季度值

A 股利润表常报告“年初至报告期”的累计金额。单季度值的规则是：

- 第一季度：`single_quarter = cumulative`；
- 后续季度：`single_quarter_t = cumulative_t - cumulative_{t-1}`；
- 跨年时重新从第一季度起算。

回测合并使用 `available_at`，而不是 `report_period`。

In [4]:
# 从累计利润得到单季度利润，并保留公告可得时间。
financials = pd.DataFrame({
    "symbol": ["DEMO_A"] * 4,
    "report_period": pd.to_datetime(["2024-03-31", "2024-06-30", "2024-09-30", "2024-12-31"]),
    "published_at": pd.to_datetime(["2024-04-25", "2024-08-20", "2024-10-28", "2025-03-25"]),
    "cumulative_profit": [10.0, 23.0, 31.0, 48.0],
})
financials["available_at"] = financials["published_at"] + pd.Timedelta(days=1)
financials["fiscal_year"] = financials["report_period"].dt.year
financials["single_quarter_profit"] = (
    financials.groupby(["symbol", "fiscal_year"])["cumulative_profit"].diff()
    .fillna(financials["cumulative_profit"])
)

display(financials)
np.testing.assert_allclose(financials["single_quarter_profit"], [10, 13, 8, 17])
assert (financials["available_at"] > financials["report_period"]).all()

,symbol,report_period,published_at,cumulative_profit,available_at,fiscal_year,single_quarter_profit
0,DEMO_A,2024-03-31,2024-04-25,10.0,2024-04-26,2024,10.0
1,DEMO_A,2024-06-30,2024-08-20,23.0,2024-08-21,2024,13.0
2,DEMO_A,2024-09-30,2024-10-28,31.0,2024-10-29,2024,8.0
3,DEMO_A,2024-12-31,2025-03-25,48.0,2025-03-26,2024,17.0


## 4. 空表也是失败，不是“没有股票”

接口即使没有抛异常，也可能成功返回空表。适配器应把它记录为 `empty` 并阻断后续计算。

In [5]:
# 把响应分成 ok、empty 和 schema_error 三种明确状态。
def inspect_response(frame, required):
    if frame.empty:
        return {"status": "empty", "rows": 0, "missing_columns": []}
    missing = sorted(set(required) - set(frame.columns))
    return {
        "status": "schema_error" if missing else "ok",
        "rows": len(frame),
        "missing_columns": missing,
    }

checks = pd.DataFrame([
    inspect_response(bars, ["date", "symbol", "close"]),
    inspect_response(pd.DataFrame(), ["date", "symbol", "close"]),
    inspect_response(bars.drop(columns="close"), ["date", "symbol", "close"]),
])
display(checks)
assert checks["status"].tolist() == ["ok", "empty", "schema_error"]

,status,rows,missing_columns
0,ok,8,[]
1,empty,0,[]
2,schema_error,8,[close]


## 方法边界与复用结论


- AkShare 聚合公开站点而非统一官方源，关键数据应与交易所和公司公告复核。
- qfq/hfq 的数值可能因后续公司行动重算；可复现研究必须保存快照。
- 指数成分、财报和行情都有各自的有效时间，不能统一用“日期”一个字段代替。


本课代码验证的是方法和数据契约，不是未来收益。迁移到真实研究前，还需重新核对来源授权、复权、时区、可得时间、交易成本和样本外边界。